read both the pdf and the online version of it

## part a

In [1]:
class LogicGate:
    def __init__(self, lbl):
        self.label = lbl
        self.output = None

    def get_label(self):
        return self.label

    def get_output(self):
        self.output = self.perform_gate_logic()
        return self.output


class BinaryGate(LogicGate):
    def __init__(self, lbl):
        LogicGate.__init__(self, lbl)
        self.pin_a = None
        self.pin_b = None

    def get_pin_a(self):
        return int(input(f"Enter pin A input for gate \
        {self.get_label()}: "))

    def get_pin_b(self):
        return int(input(f"Enter pin B input for gate \
        {self.get_label()}: "))

class AndGate(BinaryGate):
    def __init__(self, lbl):
        super().__init__(lbl)

    def perform_gate_logic(self):
        a = self.get_pin_a()
        b = self.get_pin_b()
        if a == 1 and b == 1:
            return 1
        else:
            return 0


class UnaryGate(LogicGate):
    def __init__(self, lbl):
        LogicGate.__init__(self, lbl)
        self.pin = None

    def get_pin(self):
        return int(input(f"Enter pin input for gate \
        {self.get_label()}: "))

#tryal
class OrGate(BinaryGate):
    def __init__(self, lbl):
        super().__init__(lbl)

    def perform_gate_logic(self):
        a = self.get_pin_a()
        b = self.get_pin_b()
        if a == 1 or b == 1:
            return 1
        else:
            return 0
class NotGate(UnaryGate):
    def __init__(self, lbl):
        super().__init__(lbl)

    def perform_gate_logic(self):
        a = self.get_pin()
        if a == 0: 
            return 1
        else:
            return 0

In [4]:
g1 = AndGate("G1")
print(g1.get_output() )
g2 = OrGate("G2")
print(g2.get_output() )

g3 = NotGate("G3")
g3.get_output()

Enter pin A input for gate         G1:  0
Enter pin B input for gate         G1:  0


0


Enter pin A input for gate         G2:  0
Enter pin B input for gate         G2:  0


0


Enter pin input for gate         G3:  0


1

## part b

## Understanding the `set_next_pin(self, source)` Method

In this circuit simulation, the variable **`source` is an instance of the `Connector` class**. 

Instead of passing raw binary values (`0` or `1`) directly into a gate, the code passes the entire **`Connector` object** (which acts like a physical wire). This allows gates to be dynamically chained together.

---

### 1. How `source` Gets Passed (The Chain Reaction)

The relationship begins when you create a new connection in your code. Look at the interaction between `Connector` and `LogicGate`:

```python
class Connector:
    def __init__(self, fgate, tgate):
        self.from_gate = fgate # The gate giving data
        self.to_gate = tgate   # The gate receiving data

        # The target gate calls its assignment method 
        # and passes THIS connector instance as the 'source'
        tgate.set_next_pin(self) 
```

When you run `c1 = Connector(g1, g3)` in your `main()` function:
* `tgate` represents `g3` (the `OrGate`).
* `self` represents `c1` (the `Connector`).
* `g3.set_next_pin(c1)` is executed automatically behind the scenes.

---

### 2. Line-by-Line Breakdown of `set_next_pin`

Because a `BinaryGate` has two input slots (`pin_a` and `pin_b`), this method acts like an automated switchboard to plug the incoming wire into the first available slot:

```python
def set_next_pin(self, source):
    # 1. If Pin A is empty, plug the connector into Pin A
    if self.pin_a == None:
        self.pin_a = source
    else:
        # 2. If Pin A is taken, check if Pin B is empty
        if self.pin_b == None:
            self.pin_b = source
        # 3. If both slots are full, reject the connection
        else:
            print("Cannot Connect: NO EMPTY PINS on this gate")
```

---

### 3. How the Gate Uses `source` to Fetch Data

When the gate finally calculates its logic, it needs to read values from its pins. Because `self.pin_a` now holds the `Connector` object instead of a number, it uses **Object Chaining** to pull the result:

```python
def get_pin_a(self):
    if self.pin_a == None:
        return int(input("Enter pin A input...")) # No wire attached
    else:
        # Wire attached! Follow it backward:
        # self.pin_a             -> The Connector object
        # .get_from()            -> Finds the sending gate (e.g., g1)
        # .get_output()          -> Tells that sending gate to compute its value
        return self.pin_a.get_from().get_output() 
```

### Summary of the Flow
1. **`Connector`** registers itself into a gate's empty pin slot as the **`source`**.
2. The **`Gate`** stores that connector object.
3. When computing output, the gate uses that connector to travel backwards up the circuit and ask the preceding gate for its calculated value.


In [8]:
class LogicGate:

    def __init__(self, lbl):
        self.name = lbl
        self.output = None

    def get_label(self):
        return self.name
    def get_output(self):
        self.output = self.perform_gate_logic()
        return self.output

        
class BinaryGate(LogicGate):

    def __init__(self, lbl):
        super(BinaryGate, self).__init__(lbl)
        self.pin_a = None
        self.pin_b = None
        
    def get_pin_a(self):
        if self.pin_a == None:
            return int(input("Enter pin A input for gate " + self.get_label() + ": "))
        else:
            return self.pin_a.get_from().get_output() # check the connector class

    def get_pin_b(self):
        if self.pin_b == None:
            return int(input("Enter pin B input for gate " + self.get_label() + ": "))
        else:
            return self.pin_b.get_from().get_output()

    def set_next_pin(self, source): # assuming we are connecting to two input gate
        if self.pin_a == None:
            self.pin_a = source
        else:
            if self.pin_b == None:
                self.pin_b = source
            else:
                print("Cannot Connect: NO EMPTY PINS on this gate")



class AndGate(BinaryGate):

    def __init__(self, lbl):
        BinaryGate.__init__(self, lbl)

    def perform_gate_logic(self):

        a = self.get_pin_a()
        b = self.get_pin_b()
        if a == 1 and b == 1:
            return 1
        else:
            return 0

class OrGate(BinaryGate):

    def __init__(self, lbl):
        BinaryGate.__init__(self, lbl)

    def perform_gate_logic(self):

        a = self.get_pin_a()
        b = self.get_pin_b()
        if a == 1 or b == 1:
            return 1
        else:
            return 0

class UnaryGate(LogicGate):

    def __init__(self, lbl):
        LogicGate.__init__(self, lbl)

        self.pin = None

    def get_pin(self):
        if self.pin == None:
            return int(input("Enter pin input for gate " + self.get_label() + ": "))
        else:
            return self.pin.get_from().get_output()

    def set_next_pin(self, source):
        if self.pin == None:
            self.pin = source
        else:
            print("Cannot Connect: NO EMPTY PINS on this gate")

class NotGate(UnaryGate):

    def __init__(self, lbl):
        UnaryGate.__init__(self, lbl)

    def perform_gate_logic(self):
        if self.get_pin():
            return 0
        else:
            return 1


class Connector: # this like an inteligent connection

    def __init__(self, fgate, tgate): 
# (fgate, tgate) are instance of a class that this connector class 'has a' relationship with ...Connector Has-a LogicGate
        self.from_gate = fgate
        self.to_gate = tgate

        tgate.set_next_pin(self) # perfoms a method of those instances

    def get_from(self):
        return self.from_gate  # just returns those instances

    def get_to(self):
        return self.to_gate   # just returns those instances 

class NandGate(AndGate):
    def perform_gate_logic(self):
        return int(not super().perform_gate_logic())




class NorGate(OrGate):
    def perform_gate_logic(self):
        if super().perform_gate_logic() == 1:
            return 0
        else:
            return 1       
            


def main():
    g1 = AndGate("G1")
    g2 = AndGate("G2")
    g3 = OrGate("G3")
    g4 = NotGate("G4")
    c1 = Connector(g1, g3)
    c2 = Connector(g2, g3)
    c3 = Connector(g3, g4)
    print(g4.get_output())

main()




Enter pin A input for gate G1:  1
Enter pin B input for gate G1:  1
Enter pin A input for gate G2:  1
Enter pin B input for gate G2:  1


0


In [9]:
#gx= NandGate('gx')
#gx.get_output()
g22 = NandGate("G22")
print(g22.get_output() )


g1 = AndGate("G1")
print(g1.get_output() )

Enter pin A input for gate G22:  1
Enter pin B input for gate G22:  1


0


Enter pin A input for gate G1:  1
Enter pin B input for gate G1:  1


1


## check again

In [1]:
class LogicGate:

    def __init__(self,n):
        self.name = n
        self.output = None

    def get_label(self):
        return self.name

    def get_output(self):
        self.output = self.perform_gate_logic() 
                        ## STEP_ 5: this method,perform_gate_logic(), is defined later, with relevaance to the 'children' classes..
        return self.output


class BinaryGate(LogicGate): # inherits from the logic gate

    def __init__(self,n):
        LogicGate.__init__(self,n) # # inherits from the logic gate,,,but search more why this syntax

        self.pin_a = None
        self.pin_b = None

    def get_pin_a(self):
        if self.pin_a == None:
            return int(input("Enter Pin A input for gate "+self.get_label()+"-->"))
        else:
#STEP_4 : see the connector class for more clarification on this, then see "STEP_5"
            return self.pin_a.get_from().get_output() 

    def get_pin_b(self):
        if self.pin_b == None:
            return int(input("Enter Pin B input for gate "+self.get_label()+"-->"))
# SREP_2 : a pin is assighned to an instacnce of the connector , source, thus can perfom connector methods as ie connector. get_from()
# thus connector. get_from() == self.pin_a.get_from()  
        else:
            return self.pin_b.get_from().get_output()

    def set_next_pin(self,source): # here source is intendeda to be the connector...
        if self.pin_a == None:
            self.pin_a = source       
        else:
            if self.pin_b == None:
                self.pin_b = source
            else:
                print("Cannot Connect: NO EMPTY PINS on this gate")


class AndGate(BinaryGate):

    def __init__(self,n):
        BinaryGate.__init__(self,n)

    def perform_gate_logic(self):

        a = self.get_pin_a()
        b = self.get_pin_b()
        if a==1 and b==1:
            return 1
        else:
            return 0

class OrGate(BinaryGate):

    def __init__(self,n):
        BinaryGate.__init__(self,n)

    def perform_gate_logic(self):

        a = self.get_pin_a()
        b = self.get_pin_b()
        if a ==1 or b==1:
            return 1
        else:
            return 0

class UnaryGate(LogicGate):

    def __init__(self,n):
        LogicGate.__init__(self,n)

        self.pin = None

    def get_pin(self):
        if self.pin == None:
            return int(input("Enter Pin input for gate "+self.get_label()+"-->"))
        else:
            return self.pin.get_from().get_output()

    def set_next_pin(self,source):
        if self.pin == None:
            self.pin = source 
        else:
            print("Cannot Connect: NO EMPTY PINS on this gate")


class NotGate(UnaryGate):

    def __init__(self,n):
        UnaryGate.__init__(self,n)

    def perform_gate_logic(self):
        if self.get_pin():
            return 0
        else:
            return 1


class Connector:

    def __init__(self, fgate, tgate): # (fgate, tgate ) are just insatnces of a gate
        self.from_gate = fgate
        self.to_gate = tgate

        tgate.set_next_pin(self) # STEP_1 of  automatic method of the to_gate,note it's the "to gate"
        # seee step two on the set_next_pin(source) methods in the gates, here source is an insatnce of a connector

    def get_from(self):
# STEP_3
#this is like : connector.get_from() which just returns a gate, ie the fgate which can also perfom methods ie fgat.get_output()
        return self.from_gate  

    def get_to(self):
#this is like : connector.get_to() which just returns a gate, ie the tgate which can also perfom methods
        return self.to_gate



def main():
   g1 = AndGate("G1")

   print(g1.get_output())

main()

Enter Pin A input for gate G1--> 1
Enter Pin B input for gate G1--> 1


1


In [2]:
main()

Enter Pin A input for gate G1--> 1
Enter Pin B input for gate G1--> 1


1


In [6]:
g1 = AndGate("G1_and")
g2 = NotGate("G2_not")
g3 = OrGate("G3_or")

c1 = Connector(g1, g3)
c2 = Connector(g2, g3)

print(g3.get_output())


Enter Pin A input for gate G1_and--> 0
Enter Pin B input for gate G1_and--> 0
Enter Pin input for gate G2_not--> 1


0


In [7]:
[char for word in ['cat', 'dog', 'rabbit']   for char in word]

['c', 'a', 't', 'd', 'o', 'g', 'r', 'a', 'b', 'b', 'i', 't']